# Kilat Transformer Training Demo on Tiny Shakespeare

## Replicating Andrej Karpathy's nanoGPT

This notebook is a replication of the nanoGPT concept (https://github.com/karpathy/nanoGPT) using a custom Transformer architecture for character-level language modeling on the Tiny Shakespeare dataset.

**Key Differences from nanoGPT:**
- Uses `KilatTransformer` (custom implementation with recall mechanism)
- Random sampling per epoch instead of sequential non-overlapping chunks
- Optimized generation with top-k, top-p, and repetition penalty
- Enhanced data coverage (sees all possible chunks, not just non-overlapping ones)

**References:**
- nanoGPT: https://github.com/karpathy/nanoGPT
- "Attention Is All You Need" (Vaswani et al., 2017)
- Tiny Shakespeare dataset by Andrej Karpathy

## 1. Setup and Imports

In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import urllib.request
from typing import List, Tuple
import random
import torch
from torch.utils.data import Dataset

from utils.config import KilatConfig
from data.collator import KilatDataCollator
# from data.dataset import KilatDataset
from training.trainer import KilatTrainer, TrainingArguments
from arc.model import KilatTransformer

# 2. Constants (nanoGPT-style hyperparameters)

In [2]:
BLOCK_SIZE: int = 256       # context length for training chunks
BATCH_SIZE: int = 32        # samples per batch (no grad accum — keep it simple)
PAD_TOKEN: int = 0
BOS_TOKEN: int = 1
EOS_TOKEN: int = 2
STRIDE: int = 32            # stride for sliding window (12.5% of BLOCK_SIZE)

# Tiny Shakespeare URL (Karpathy)
DATA_URL: str = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH: str = "tinyshakespeare.txt"

# 3. Character-level Tokenizer (same as nanoGPT)

In [3]:
class CharTokenizer:
    def __init__(self, text: str) -> None:
        chars = sorted(list(set(text)))
        self.stoi: dict[str, int] = {ch: i + 3 for i, ch in enumerate(chars)}  # 0,1,2 reserved
        self.itos: dict[int, str] = {i: ch for ch, i in self.stoi.items()}

        self.stoi["<PAD>"] = PAD_TOKEN
        self.stoi["<BOS>"] = BOS_TOKEN
        self.stoi["<EOS>"] = EOS_TOKEN
        self.itos[PAD_TOKEN] = "<PAD>"
        self.itos[BOS_TOKEN] = "<BOS>"
        self.itos[EOS_TOKEN] = "<EOS>"

        self.vocab_size = len(self.stoi)
        self.pad_token_id = PAD_TOKEN
        self.bos_token_id = BOS_TOKEN
        self.eos_token_id = EOS_TOKEN

    def encode(self, text: str) -> list[int]:
        return [self.stoi.get(c, PAD_TOKEN) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.itos.get(i, "?") for i in ids)

# 4. Dataset Helpers (Enhanced version of nanoGPT's data loading)

In [4]:
def download_tiny_shakespeare() -> str:
    if not os.path.exists(DATA_PATH):
        print(f"Downloading Tiny Shakespeare dataset...")
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        print("Download complete.")
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()
    print(f"Dataset length: {len(text):,} characters")
    return text

In [5]:
class ShakespeareDataset(Dataset):    
    def __init__(self, data: torch.Tensor, block_size: int = BLOCK_SIZE):
        self.data = data
        self.block_size = block_size
        self.max_start = len(data) - block_size
        
    def __len__(self) -> int:
        return max(1, self.max_start)
    
    def __getitem__(self, idx: int) -> dict:
        if self.max_start > 0:
            start = torch.randint(0, self.max_start, (1,)).item()
        else:
            start = 0
            
        chunk = self.data[start:start + self.block_size]
        return {"input_ids": chunk.tolist()}

In [6]:

def prepare_datasets(
    text: str, tokenizer: CharTokenizer, train_frac: float = 0.9
) -> Tuple[ShakespeareDataset, ShakespeareDataset]:
    data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
    
    n = int(train_frac * len(data))
    train_data = data[:n]
    eval_data = data[n:]
    
    print(f"Total tokens: {len(data):,}")
    print(f"Train tokens: {len(train_data):,} ({(len(train_data)/len(data))*100:.1f}%)")
    print(f"Eval tokens:  {len(eval_data):,} ({(len(eval_data)/len(data))*100:.1f}%)")
    print(f"Possible train chunks: {max(0, len(train_data) - BLOCK_SIZE):,}")
    print(f"Possible eval chunks:  {max(0, len(eval_data) - BLOCK_SIZE):,}")
    
    train_dataset = ShakespeareDataset(train_data, block_size=BLOCK_SIZE)
    eval_dataset = ShakespeareDataset(eval_data, block_size=BLOCK_SIZE)
    return train_dataset, eval_dataset

# 5. Model Builder (KilatTransformer - Custom Architecture)

In [ ]:
def build_model(vocab_size: int) -> KilatTransformer:
    config = KilatConfig(
        vocab_size=vocab_size,
        n_embd=384,
        n_layer=6,
        n_head=6,
        recall_ratio=0.5,
        latent_dim=32,
        ffn_mode="dense",
        ff_mult=8 / 3,
        embd_drop=0.1,
        attn_drop=0.1,
        ffn_dropout=0.1,
        resid_drop=0.1,
        pad_token_id=PAD_TOKEN,
        bos_token_id=BOS_TOKEN,
        eos_token_id=EOS_TOKEN,
        tie_word_embeddings=True,
    )

    model = KilatTransformer(config)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Vocabulary size:     {vocab_size:,}")
    print(f"Total parameters:    {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return model

# 6. Text Generation (Enhanced version with advanced sampling)

In [ ]:

@torch.inference_mode()
def generate(
    model: KilatTransformer,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 500,
    temperature: float = 0.7,
    top_k: int = 15,                 
    top_p: float = 0.85,            
    repetition_penalty: float = 1.15, 
    device: str = "cpu",
) -> str:
    model.eval()
    input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        if input_ids.size(1) > BLOCK_SIZE:
            input_ids = input_ids[:, -BLOCK_SIZE:]

        outputs = model(input_ids, return_dict=True)
        logits = outputs.logits[:, -1, :].clone()  

        if temperature == 0.0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            if repetition_penalty != 1.0:
                for score_idx in range(logits.size(0)):
                    # Ambil token unik yang sudah digenerasikan sejauh ini
                    generated_tokens = set(input_ids[score_idx].tolist())
                    for token_id in generated_tokens:
                        # Jika logit > 0, turunkan nilainya; jika < 0, buat semakin negatif
                        if logits[score_idx, token_id] > 0:
                            logits[score_idx, token_id] /= repetition_penalty
                        else:
                            logits[score_idx, token_id] *= repetition_penalty

            logits = logits / temperature
            
            if top_k > 0:
                top_k_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                kth_values = top_k_values[..., -1, None]
                logits[logits < kth_values] = float('-inf')
            
            if top_p > 0.0 and top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
                cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                logits[indices_to_remove] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == EOS_TOKEN:
            break

    return tokenizer.decode(input_ids[0].tolist())

## 7. Main Training Loop

### Mixed Precision Training

To make training faster and more memory efficient, we use mixed precision training:
- **bfloat16 (bf16)**: best for modern GPUs (A100, H100, RTX 4090, etc.) — provides better numerical stability  
- **float16 (fp16)**: good for older GPUs (V100, RTX 3090, T4, etc.) — faster but may have gradient underflow  
- **float32 (fp32)**: fallback for CPUs or when mixed precision causes issues — slower but most stable  

**Note:** if you don't have a compatible GPU, training will fall back to CPU, which only supports fp32.

In [9]:
def main() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    text = download_tiny_shakespeare()
    tokenizer = CharTokenizer(text)
    
    print(f"Vocabulary size: {tokenizer.vocab_size} unique characters (+3 special tokens)")
    print(f"Special tokens: <PAD>={PAD_TOKEN}, <BOS>={BOS_TOKEN}, <EOS>={EOS_TOKEN}")
    print()

    train_dataset, eval_dataset = prepare_datasets(text, tokenizer, train_frac=0.9)
    
    print(f"\nTraining dataset size (epochs worth): {len(train_dataset):,} unique chunks")
    print(f"Evaluation dataset size: {len(eval_dataset):,} unique chunks")
    print(f"Each epoch will sample ~{min(len(train_dataset), 5000):,} random chunks")
    print()

    data_collator = KilatDataCollator(
        pad_token_id=PAD_TOKEN,
        max_length=BLOCK_SIZE,
        ignore_index=-100,
    )

    model = build_model(tokenizer.vocab_size)
    model.to(device)
    total_steps = 5000
    tokens_per_step = BATCH_SIZE * BLOCK_SIZE
    total_tokens_seen = total_steps * tokens_per_step
    total_unique_tokens = len(train_dataset.data) - BLOCK_SIZE
    
    print(f"\nTraining statistics:")
    print(f"  Total training steps: {total_steps:,}")
    print(f"  Tokens per step: {tokens_per_step:,}")
    print(f"  Total tokens processed: {total_tokens_seen:,}")
    print(f"  Unique tokens available: {total_unique_tokens:,}")
    print(f"  Coverage: {min(100.0, (total_tokens_seen / total_unique_tokens) * 100):.1f}% of possible chunks")
    print()

    training_args = TrainingArguments(
        output_dir="./tmp_kilat",
        save_checkpoints=False,
        training_mode="steps",
        max_steps=total_steps,
        learning_rate=3e-4,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_steps=200,
        logging_steps=100,
        eval_steps=500,
        save_steps=1_000_000_000,
        save_total_limit=0,
        early_stopping_patience=0,
        precision="bf16" if device == "cuda" else "fp32",
        report_to="none",
        run_name="kilat-shakespeare",
        seed=1337,
    )

    trainer = KilatTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )
    trainer.train()

    print("\n" + "=" * 60)
    print("Generating samples with optimized sampling strategy...")
    print("=" * 60 + "\n")

    prompts = [
        "First Citizen:",
        "QUEEN MARGARET:",
        "To be, or not to be,",
        "    ",
        "ROMEO:",
        "HAMLET:",
    ]

    for prompt in prompts:
        print(f"--- Prompt: \"{prompt.strip()}\" ---")
        generated = generate(
            model, tokenizer, prompt,
            max_new_tokens=500,
            temperature=0.7,
            top_k=15,            
            top_p=0.85,           
            repetition_penalty=1.15, 
            device=device,
        )
        print(generated)
        print()
        print("-" * 60)
        print()

    print("Done.")


In [10]:
main()

Using device: cuda

Dataset length: 1,115,394 characters
Vocabulary size: 68 unique characters (+3 special tokens)
Special tokens: <PAD>=0, <BOS>=1, <EOS>=2

Total tokens: 1,115,394
Train tokens: 1,003,854 (90.0%)
Eval tokens:  111,540 (10.0%)
Possible train chunks: 1,003,598
Possible eval chunks:  111,284

Training dataset size (epochs worth): 1,003,598 unique chunks
Evaluation dataset size: 111,284 unique chunks
Each epoch will sample ~5,000 random chunks

Vocabulary size:     68
Total parameters:    11,353,746
Trainable parameters: 11,353,746

Training statistics:
  Total training steps: 5,000
  Tokens per step: 8,192
  Total tokens processed: 40,960,000
  Unique tokens available: 1,003,598
  Coverage: 100.0% of possible chunks


KilatTransformer Training
Output dir:          ./tmp_kilat
Save checkpoints:    False
Training mode:       steps
Total target steps:  5,000
Batch size (per GPU):32
Gradient accum:      1
Effective batch:     32
Learning rate:       3.00e-04
Warmup steps:   

Training (steps):   2%|▏         | 101/5000 [00:18<10:25,  7.83step/s, loss=2.2908, ppl=9.9, lr=1.51e-04] 

[01:37:30] Step:    100/ 5,000 | Loss:   2.3269 | PPL:     10.2 | LR: 1.50e-04 | Grad:   0.91 | Speed:    5.4 st/s | Remaining:   15.0 min


Training (steps):   4%|▍         | 201/5000 [00:30<09:06,  8.78step/s, loss=2.0076, ppl=7.4, lr=3.00e-04] 

[01:37:42] Step:    200/ 5,000 | Loss:   2.0430 | PPL:      7.7 | LR: 3.00e-04 | Grad:   0.92 | Speed:    6.5 st/s | Remaining:   12.3 min


...

[01:47:31] Step:  1,900/ 5,000 | Loss:   1.2276 | PPL:      3.4 | LR: 2.16e-04 | Grad:   1.06 | Speed:    3.1 st/s | Remaining:   16.8 min


Training (steps):  40%|████      | 2000/5000 [10:30<05:41,  8.78step/s, loss=1.2609, ppl=3.5, lr=2.07e-04]

[01:47:42] Step:  2,000/ 5,000 | Loss:   1.2609 | PPL:      3.5 | LR: 2.07e-04 | Grad:   0.99 | Speed:    3.2 st/s | Remaining:   15.8 min


Training (steps):  40%|████      | 2000/5000 [12:40<19:00,  2.63step/s, loss=1.2609, ppl=3.5, lr=2.07e-04]



----------------------------------------
Evaluation Summary
----------------------------------------
Validation Loss: 1.6557
Validation PPL:  5.2
Best Val Loss:   1.6535
Step:            2000
----------------------------------------


[EarlyStopping] Eval loss 1.6557 did not improve over best 1.6535. Counter: 1/0

Early stopping triggered at step 2000

Training Summary
Total steps:       2,000
Total time:        12.7 min (760.2 sec)
Avg steps/sec:     2.6
Best eval loss:    1.6535
Best eval PPL:     5.2
Output directory:  ./tmp_kilat


Generating samples with optimized sampling strategy...

--- Prompt: "First Citizen:" ---
.

Clown:
I have been commanded to my heart it? An your brother's face,
And you cannot temperation. Lord Angelo,
To the part the court of the sway of the time
To the secret of the hands.

DUKE VINCENTIO:
Be look upon him in the pettite of all thing on the
p

------------------------------------------------------------

--- Prompt: "QUEEN MARGARET:" ---
I:
What will 